# License Plate OCR with SmolVLM

Vision-language model based text extraction from a license plate image, using `HuggingFaceTB/SmolVLM-Instruct`.

## 1. Setup & Imports

In [1]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 99.5 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1


In [2]:
# Ambil HF_TOKEN dari Colab Secrets (opsional, untuk rate limit lebih tinggi), fallback input manual
from google.colab import userdata
import getpass

try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Masukkan HF_TOKEN (opsional, kosongkan lalu Enter jika tidak ada): ")

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)

In [3]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Install flash_attn if using CUDA for Flash Attention 2
# if DEVICE == "cuda":
#     !pip install flash-attn --no-build-isolation

processor = AutoProcessor.from_pretrained("HuggingFaceTB/SmolVLM-Instruct")
model = AutoModelForImageTextToText.from_pretrained("HuggingFaceTB/SmolVLM-Instruct",
                                                torch_dtype=torch.bfloat16,
                                                _attn_implementation="eager").to(DEVICE)

processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/7.45k [00:00<?, ?B/s]

[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 128002. This may result in unexpected behavior.


...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 4.49GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/657 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

## 2. Load Image and Build Prompt

In [4]:
from google.colab import files
from PIL import Image
from transformers.image_utils import load_image

# Upload gambar plat nomor secara manual (menghindari timeout dari fetch URL eksternal)
uploaded = files.upload()
image_path = list(uploaded.keys())[0]
image1 = load_image(image_path)

# Create input messages
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "You are an expert license plate extractor agent. Extract this text with only the text, no additional characters (only the OCR result from this image). Return it in JSON format."}
        ]
    },
]

# Prepare inputs
prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=prompt, images=[image1], return_tensors="pt")
inputs = inputs.to(DEVICE)

Saving Plat-Mobil.jpeg to Plat-Mobil.jpeg


In [ ]:
import matplotlib.pyplot as plt

# Tampilkan gambar yang diupload, supaya input dan hasil ekstraksinya bisa dibandingkan langsung
plt.figure(figsize=(6, 6))
plt.imshow(image1)
plt.axis('off')
plt.title(f"Input: {image_path}")
plt.show()

## 3. Run Inference

In [5]:
# Generate outputs
generated_ids = model.generate(**inputs, max_new_tokens=500)

# Only decode the newly generated tokens, not the echoed prompt
new_tokens = generated_ids[:, inputs['input_ids'].shape[1]:]
generated_texts = processor.batch_decode(new_tokens, skip_special_tokens=True)

raw_output = generated_texts[0].strip()
print("Raw model output:")
print(raw_output)

Raw model output:
{
    "text": "B 4213 L"
}


## Parsing and Validating the Output

A VLM prompted to return JSON doesn't always do so reliably — it can wrap the JSON in extra text, use inconsistent quoting, or occasionally miss the format entirely. This step actually parses the output and reports clearly if parsing fails, instead of assuming the raw text is always valid JSON.

In [6]:
import json
import re

def extract_json(text):
    """Find and parse the first {...} block in the model's output."""
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if not match:
        return None, "No JSON object found in model output."
    try:
        return json.loads(match.group(0)), None
    except json.JSONDecodeError as e:
        return None, f"Found a JSON-like block but it failed to parse: {e}"

result, error = extract_json(raw_output)

if result is not None:
    print("Parsed result:", result)
    plate_text = result.get("text", "(no 'text' key in output)")
    print("Extracted plate text:", plate_text)
else:
    print("Could not parse a valid JSON result.")
    print("Reason:", error)
    print("Falling back to raw output above for manual inspection.")

Parsed result: {'text': 'B 4213 L'}
Extracted plate text: B 4213 L


---

*Hands-on material: rubythalib.ai*